In [ ]:
#============= OKOK - photo ==========

import asyncio
import json
import random
import traceback
import nodriver as uc

TARGET_PAGE_URL = "https://www.facebook.com/mocseafood/"


async def lay_20_anh_tu_tab_photos(fb_tab):
    """
    Click vào tab "Photos" của page, kéo xuống 2 lần,
    rồi lấy 20 ảnh đầu tiên (src chứa 'scontent').
    """

    print("📷 Tìm tab Photos...")

    try:
        clicked = await fb_tab.evaluate(
            """
            (() => {
                const links = Array.from(
                    document.querySelectorAll("a")
                );

                const target = links.find((a) => {
                    const t = (a.innerText || "").trim();

                    return (
                        t === "Photos" ||
                        t === "Ảnh" ||
                        t === "Hình ảnh" ||
                        (
                            a.href &&
                            a.href.includes("/photos")
                        )
                    );
                });

                if (target) {
                    target.click();
                    return true;
                }

                return false;
            })();
            """
        )
    except Exception as e:
        print("Lỗi khi tìm tab Photos:", e)
        clicked = False

    if not clicked:
        print("❌ Không tìm thấy tab Photos.")
        return []

    print("✅ Đã click tab Photos. Đợi tải...")

    await asyncio.sleep(random.uniform(1.8, 2.6))

    print("📜 Kéo xuống 2 lần để load thêm ảnh...")

    for round_idx in range(2):
        print(f"🔄 Scroll round {round_idx + 1}/2")

        try:
            await fb_tab.evaluate(
                """
                window.scrollBy(0, 3000);
                """
            )
        except Exception:
            pass

        await asyncio.sleep(random.uniform(1.5, 2.5))

    print("🖼️ Thu thập danh sách ảnh...")

    try:
        urls = await fb_tab.evaluate(
            """
            (() => {
                const imgs = Array.from(
                    document.querySelectorAll("img")
                );

                const srcs = imgs
                    .map((img) => img.src)
                    .filter(
                        (src) =>
                            src &&
                            src.includes("scontent")
                    );

                return Array.from(
                    new Set(srcs)
                ).slice(0, 20);
            })();
            """
        )
    except Exception as e:
        print("Lỗi khi lấy ảnh:", e)
        urls = []

    urls = urls or []

    print(f"✅ Lấy được {len(urls)} ảnh.")

    with open(
        "fb_photos.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(urls, f, ensure_ascii=False, indent=2)

    print("💾 Đã lưu fb_photos.json")

    return urls


async def inspect_html_and_click():
    browser = None

    try:
        print("🚀 Mở trình duyệt...")

        browser = await uc.start(
            headless=False,
            icon_mode=1
        )

        print("🔍 Vào Google...")

        page = await browser.get(
            "https://www.google.com/search?q=site:facebook.com/mocseafood"
        )

        print("⏳ Đợi kết quả xuất hiện...")

        target_element = None

        for _ in range(30):
            try:
                target_element = await page.select(
                    f'a[href="{TARGET_PAGE_URL}"]'
                )

                if target_element:
                    break

            except Exception:
                pass

            await asyncio.sleep(0.2)

        if not target_element:
            print("❌ Không tìm thấy link.")
            return

        print("✅ Đã tìm thấy link.")

        try:
            await target_element.scroll_into_view()
        except Exception:
            pass

        await asyncio.sleep(
            random.uniform(0.34, 0.8)
        )

        print("🖱️ Click Facebook...")

        await target_element.click()

        await asyncio.sleep(
            random.uniform(1.5, 2.5)
        )

        print("🔎 Kiểm tra tabs...")

        fb_tab = None

        for tab in browser.tabs:
            try:
                url = str(tab.url)

                print("TAB:", url)

                if "facebook.com" in url.lower():
                    fb_tab = tab

            except Exception:
                pass

        if not fb_tab:
            print("❌ Không tìm thấy tab Facebook.")
            return

        print("🎉 Đã tìm thấy Facebook.")

        print("📜 Scroll + Expand Posts + Expand Comments...")

        for round_idx in range(2):

            print(f"🔄 Round {round_idx + 1}/10")

            # =====================================================
            # SCROLL
            # =====================================================

            try:
                await fb_tab.evaluate(
                    """
                    window.scrollBy(0, 3000);
                    """
                )
            except Exception:
                pass

            await asyncio.sleep(
                random.uniform(1.5, 2.5)
            )

        print("📸 Lấy HTML...")

        html = await fb_tab.get_content()

        with open(
            "facebook_page.html",
            "w",
            encoding="utf-8"
        ) as f:
            f.write(html)

        print(
            f"💾 Đã lưu facebook_page.html "
            f"({len(html):,} ký tự)"
        )

        # =====================================================
        # CLICK "SEE MORE"
        # CLICK "VIEW MORE COMMENTS"
        # =====================================================

        try:

            await fb_tab.evaluate(
                """
                (() => {

                    const targets = [
                        "See more",
                        "View more comments",
                        "View previous comments",
                        "View more replies",

                        "Xem thêm",
                        "Xem thêm bình luận",
                        "Xem phản hồi",
                        "Xem thêm phản hồi"
                    ];

                    const all = document.querySelectorAll("*");

                    for (const el of all) {

                        const text =
                            (el.innerText || "")
                            .trim();

                        if (
                            targets.includes(text)
                        ) {

                            try {
                                el.click();
                            } catch(e) {}

                        }
                    }

                })();
                """
            )

        except Exception:
            pass

        await asyncio.sleep(
            random.uniform(0.6, 1.2)
        )

        print("✅ Hoàn tất expand.")

        # ==========================================
        # CHỨC NĂNG MỚI: VÀO TAB PHOTOS, LẤY 20 ẢNH
        # ==========================================

        anh_list = await lay_20_anh_tu_tab_photos(fb_tab)
        print("📋 Danh sách ảnh:", anh_list)

        # ==========================================
        # DEBUG DIALOG
        # ==========================================
        await asyncio.sleep(2)
        try:

            dialog_count = await fb_tab.evaluate("""
            document.querySelectorAll('[role="dialog"]').length
            """)

            print("FOUND DIALOGS:", dialog_count)

            popup_html = await fb_tab.evaluate("""
            (() => {
                const dialogs =
                    document.querySelectorAll('[role="dialog"]');

                if (!dialogs.length)
                    return "NO_DIALOG";

                return dialogs[dialogs.length - 1].outerHTML;
            })()
            """)

            print("POPUP SIZE:", len(popup_html))

            with open(
                "popup_only.html",
                "w",
                encoding="utf-8"
            ) as f:
                f.write(popup_html)

            print("💾 Saved popup_only.html")

        except Exception as e:

            print("Popup debug error:", e)


    except Exception as e:
        print("❌ Lỗi:", e)
        traceback.print_exc()

    finally:
        print("⏳ Giữ trình duyệt 30s...")
        #await asyncio.sleep(1)

        try:
            if browser:
                browser.stop()
        except Exception:
            pass

if __name__ == "__main__":
    await inspect_html_and_click()